# Consulta de rentabilidades de bonos

Elige `api`, un `ticker` de su fila y el intervalo de fechas en la celda de configuración. `M` significa meses y `Y`, años: `JP10Y` corresponde al plazo de 10 años de Japón.

Referencias admitidas por los archivos del proyecto (no garantiza datos para cualquier fecha):

| Valor de `api` | Fuente | Valores de `ticker` admitidos |
| --- | --- | --- |
| `us_treasury` | Tesoro de Estados Unidos | `US1M`, `US2M`, `US3M`, `US4M`, `US6M`, `US1Y`, `US2Y`, `US3Y`, `US5Y`, `US7Y`, `US10Y`, `US20Y`, `US30Y` |
| `bce` | BCE, zona euro AAA | `EU3M`, `EU4M`, `EU6M`, `EU1Y`, `EU2Y`, `EU3Y`, `EU5Y`, `EU7Y`, `EU10Y`, `EU20Y`, `EU30Y` |
| `japan` | Ministerio de Finanzas de Japón | `JP1Y`, `JP2Y`, `JP3Y`, `JP4Y`, `JP5Y`, `JP6Y`, `JP7Y`, `JP8Y`, `JP9Y`, `JP10Y`, `JP15Y`, `JP20Y`, `JP25Y`, `JP30Y`, `JP40Y` |
| `uk` | Bank of England | `UK5Y`, `UK10Y`, `UK20Y` |

Ejemplos de parejas válidas: `us_treasury` / `US10Y`, `bce` / `EU10Y`, `japan` / `JP10Y`, `uk` / `UK10Y`.

Las series contienen rentabilidades en porcentaje, no precios. Ejecuta las celdas en orden. La consulta requiere conexión a internet, `pandas` y `requests`.


In [1]:
from pathlib import Path
import sys

raiz = Path.cwd()

if raiz.name == "tests":
    raiz = raiz.parent

carpeta_modelos = raiz / "src"

if not carpeta_modelos.is_dir():
    raise FileNotFoundError(
        f"No se encuentra la carpeta: {carpeta_modelos}"
    )

sys.path.insert(0, str(carpeta_modelos))

In [2]:
from financial_toolkit.us_treasury_api import USTreasuryAPI
from financial_toolkit.bce_bond_api import BCEBondAPI
from financial_toolkit.jp_bond_api import JapanBondAPI
from financial_toolkit.uk_bond_api import UKBondAPI
from financial_toolkit.bond_normalizer import BondNormalizer

entrada = {
    "ticker": "US10Y",
    "type": "bond",
    "api": "us_treasury",
    "start_date": "2025-10-01",
    "end_date": "2026-10-30",
}

fetchers = {
    "us_treasury": USTreasuryAPI,
    "bce": BCEBondAPI,
    "japan": JapanBondAPI,
    "uk": UKBondAPI,
}

if entrada["type"] != "bond":
    raise ValueError("Este ejemplo solo admite el tipo bond.")

if entrada["api"] not in fetchers:
    raise ValueError("API no admitida.")

# Comprobar la pareja API/ticker antes de descargar datos.
entrada["api"] = entrada["api"].strip().lower()
entrada["ticker"] = entrada["ticker"].strip().upper()
if entrada["api"] not in fetchers:
    raise ValueError(f"API no admitida. Opciones: {', '.join(fetchers)}")
opciones = fetchers[entrada["api"]].SERIES_MAP
if entrada["ticker"] not in opciones:
    raise ValueError(
        f"El ticker {entrada['ticker']} no corresponde a {entrada['api']}. "
        f"Opciones: {', '.join(opciones)}"
    )


In [3]:
# 1. Crear el fetcher de la fuente elegida.
fetcher = fetchers[entrada["api"]]()


In [4]:
# 2. Descargar los datos originales.
datos_originales = fetcher.get_data(
    ticker=entrada["ticker"],
    start_date=entrada["start_date"],
    end_date=entrada["end_date"],
)


In [5]:
# 3. Normalizar, limpiar y construir el objeto Bond.
normalizer = BondNormalizer()
bono = normalizer.normalize(
    data=datos_originales,
    ticker=entrada["ticker"],
    source=entrada["api"],
    start_date=entrada["start_date"],
    end_date=entrada["end_date"],
)


In [6]:
# 4. Mostrar la información del objeto.
print(bono.name)
print(f"Identificador: {bono.ticker}")
print(f"País o zona: {bono.country}")
print(f"Plazo en años: {bono.maturity_years}")
print(f"Fuente: {bono.source}")
print("\nHistórico de rentabilidades (%):")
print(bono.series.ytm)
print(f"\nTotal de observaciones: {len(bono.series.ytm)}")
print(f"Primera fecha: {bono.series.date[0]:%Y-%m-%d}")
print(f"Última fecha: {bono.series.date[-1]:%Y-%m-%d}")


US Treasury: rentabilidad a la par a 10Y (USD)
Identificador: US10Y
País o zona: US
Plazo en años: 10
Fuente: us_treasury

Histórico de rentabilidades (%):
date
2025-10-01    4.12
2025-10-02    4.10
2025-10-03    4.13
2025-10-06    4.18
2025-10-07    4.14
              ... 
2026-10-26    5.00
2026-10-27    4.99
2026-10-28    5.05
2026-10-29    4.98
2026-10-30    4.94
Name: US10Y, Length: 271, dtype: float64

Total de observaciones: 271
Primera fecha: 2025-10-01
Última fecha: 2026-10-30


In [ ]:
# 5. Ejecutar los métodos de BondSeries.

print(f"Última rentabilidad: {bono.series.current_yield():.3f} %")
print(f"Rentabilidad máxima del intervalo: {bono.series.yield_max():.3f} %")
print(f"Rentabilidad mínima del intervalo: {bono.series.yield_min():.3f} %")
print(f"Rentabilidad media: "f"{bono.series.yield_mean():.3f} %")
print(f"Cambio total del intervalo: "f"{bono.series.yield_change_total_bps():.2f} puntos básicos")
print(f"Volatilidad de las rentabilidades (std): "f"{bono.series.yield_volatility():.3f} puntos porcentuales")
print("Últimos cambios entre observaciones en puntos básicos:")
print(bono.series.yield_change_bps().tail())



Última rentabilidad: 4.940 %
Rentabilidad máxima del intervalo: 5.310 %
Rentabilidad mínima del intervalo: 3.970 %

Últimos cambios entre observaciones en puntos básicos:
date
2026-10-26    0.0
2026-10-27   -1.0
2026-10-28    6.0
2026-10-29   -7.0
2026-10-30   -4.0
Name: US10Y, dtype: float64
Rentabilidad media: 4.431 %
Cambio total del intervalo: 82.00 puntos básicos
Volatilidad de las rentabilidades (std): 0.341 puntos porcentuales
